# 2D TFI Ground-State Energy with DMRG

This notebook only estimates the ground-state energy of the 2D transverse-field Ising model on a small periodic lattice.

In [ ]:
using ITensors
using ITensorMPS

# ------------------- User parameters -------------------
const Lx = 4
const Ly = 4
const N = Lx * Ly

const h = 3.0
const J = 1.0
const pbc = true

# ------------------- 2D lattice helpers -------------------
@inline site_index(x::Int, y::Int) = (y - 1) * Lx + x

function build_bonds(Lx::Int, Ly::Int; pbc::Bool=true)
    bonds = Vector{NTuple{2, Int}}()

    for y in 1:Ly
        for x in 1:(Lx - 1)
            push!(bonds, (site_index(x, y), site_index(x + 1, y)))
        end
        if pbc
            push!(bonds, (site_index(Lx, y), site_index(1, y)))
        end
    end

    for x in 1:Lx
        for y in 1:(Ly - 1)
            push!(bonds, (site_index(x, y), site_index(x, y + 1)))
        end
        if pbc
            push!(bonds, (site_index(x, Ly), site_index(x, 1)))
        end
    end

    bonds = map(b -> (min(b[1], b[2]), max(b[1], b[2])), bonds)
    sort!(bonds)
    unique!(bonds)
    return bonds
end

bonds = build_bonds(Lx, Ly; pbc=pbc)
sites = siteinds("S=1/2", N)

# ------------------- Hamiltonian -------------------
os = OpSum()
for i in 1:N
    os += -h, "X", i
end
for (i, j) in bonds
    os += -J, "Z", i, "Z", j
end
H = MPO(os, sites)

# ------------------- Initial state -------------------
psi0 = MPS(sites)
for i in 1:N
    psi0[i] = ITensor(sites[i])
    psi0[i][1] = 1.0
    psi0[i][2] = 1.0
end
normalize!(psi0)

# ------------------- DMRG parameters -------------------
nsweeps = 20
maxdim = [500]
cutoff = [1e-10]
noise = [1e-6, 1e-7, 1e-8, 0.0]

# ------------------- Ground-state energy -------------------
energy, psi = dmrg(H, psi0; nsweeps, maxdim, cutoff, noise)

println("------ Results ------")
println("Model: 2D TFI")
println("Lx x Ly = $(Lx) x $(Ly), N = $N, PBC = $pbc")
println("Parameters: h = $h, J = $J")
println("Ground-state energy E0 = $energy")
println("Energy per site e0 = $(energy / N)")
